# MLP 2x100 ReLU + Dropout

Dos capas de 100 neuronas con Dropout para reducir sobreajuste.

Entrena esta arquitectura para las 16 combinaciones de ventanas y guarda resultados en `data/mlp/mlp_2x100_dropout.csv`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "util.py").exists():
    for parent in Path.cwd().parents:
        if (parent / "util.py").exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import keras
from keras.models import Sequential
from keras.layers import Dense, Input
from keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error

from util import (
    get_train_test,
    RANDOM_SEED,
    compare_to_benchmark,
    plot_benchmark_comparison,
    configure_mlflow,
    log_keras_grid_run,
)

keras.utils.set_random_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
from keras.layers import Dropout


In [2]:
MODEL_NAME = "mlp_2x100_dropout"
EXPERIMENT_NAME = "MLP"
LOG_MODEL_ARTIFACT = False
ARCHITECTURE_PARAMS = {
    'hidden_layers': 2,
    'neurons': 100,
    'activation': 'relu',
    'regularization': 'dropout',
    'dropout_rate': 0.2,
}
mlflow = configure_mlflow(EXPERIMENT_NAME)
RESULTS_PATH = PROJECT_ROOT / "data" / "mlp" / f"{MODEL_NAME}.csv"
HISTORY_PATH = PROJECT_ROOT / "data" / "mlp" / f"{MODEL_NAME}_history.csv"

input_windows = [5, 10, 30, 90]
output_windows = [1, 5, 30, 90]

EPOCHS = 200
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
VALIDATION_SPLIT = 0.1

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
print("results:", RESULTS_PATH)
print("history:", HISTORY_PATH)


results: /Users/jchulvi/projects/Neural-Networks-Forecasting/data/mlp/mlp_2x100_dropout.csv
history: /Users/jchulvi/projects/Neural-Networks-Forecasting/data/mlp/mlp_2x100_dropout_history.csv


## Definicion de la red

In [3]:
def build_model(input_dim, output_dim):
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    model.add(Dense(100, activation="relu"))
    model.add(Dropout(0.2))
    model.add(Dense(100, activation="relu"))
    model.add(Dropout(0.2))
    model.add(Dense(output_dim))
    model.compile(loss="mean_absolute_error", optimizer=Adam(learning_rate=LEARNING_RATE))
    return model


## Entrenamiento en grid de ventanas

In [4]:
results = []
history_rows = []

for in_w in input_windows:
    for out_w in output_windows:
        d = get_train_test(input_window_size=in_w, output_window_size=out_w)

        X_train = d.X_train.reshape(d.X_train.shape[0], -1)
        X_test = d.X_test.reshape(d.X_test.shape[0], -1)

        keras.utils.set_random_seed(RANDOM_SEED)
        model = build_model(input_dim=X_train.shape[1], output_dim=d.y_train.shape[1])


        hist = model.fit(
            X_train,
            d.y_train,
            validation_split=VALIDATION_SPLIT,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=0,
            shuffle=False,
        )

        y_pred_train = model.predict(X_train, verbose=0)
        y_pred_test = model.predict(X_test, verbose=0)

        row = {
            "model_name": MODEL_NAME,
            "input_window": in_w,
            "output_window": out_w,
            "MAE_train": mean_absolute_error(d.y_train, y_pred_train),
            "MAE_val": min(hist.history["val_loss"]),
            "MAE_test": mean_absolute_error(d.y_test, y_pred_test),
            "epochs": len(hist.history["loss"]),
            "n_params": model.count_params(),
        }
        results.append(row)

        for epoch, (loss, val_loss) in enumerate(
            zip(hist.history["loss"], hist.history["val_loss"]), start=1
        ):
            history_rows.append({
                "model_name": MODEL_NAME,
                "input_window": in_w,
                "output_window": out_w,
                "epoch": epoch,
                "loss": loss,
                "val_loss": val_loss,
            })

        run_name = f"{MODEL_NAME}_input{in_w}_output{out_w}"
        log_keras_grid_run(
            mlflow=mlflow,
            model=model,
            history=hist,
            run_name=run_name,
            model_name=MODEL_NAME,
            input_window=in_w,
            output_window=out_w,
            metrics_row=row,
            train_shape=X_train.shape,
            test_shape=X_test.shape,
            output_dim=d.y_train.shape[1],
            batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            validation_split=VALIDATION_SPLIT,
            extra_params=ARCHITECTURE_PARAMS,
            log_model_artifact=LOG_MODEL_ARTIFACT,
        )

        results_df = pd.DataFrame(results)
        history_df = pd.DataFrame(history_rows)
        results_df.to_csv(RESULTS_PATH, index=False)
        history_df.to_csv(HISTORY_PATH, index=False)

        print(
            f"input={in_w:>3} output={out_w:>3} | "
            f"train={row['MAE_train']:.6f} val={row['MAE_val']:.6f} "
            f"test={row['MAE_test']:.6f} epochs={row['epochs']} "
            f"params={row['n_params']}"
        )

results_df = pd.DataFrame(results)
history_df = pd.DataFrame(history_rows)
results_df


input=  5 output=  1 | train=0.010278 val=0.009028 test=0.012710 epochs=200 params=24023


input=  5 output=  5 | train=0.004771 val=0.004166 test=0.005774 epochs=200 params=24023


input=  5 output= 30 | train=0.002081 val=0.001764 test=0.002417 epochs=200 params=24023


input=  5 output= 90 | train=0.001277 val=0.001023 test=0.001414 epochs=200 params=24023


input= 10 output=  1 | train=0.009759 val=0.009017 test=0.012971 epochs=200 params=35523


...

input= 90 output=  1 | train=0.008984 val=0.009032 test=0.012829 epochs=200 params=219523


input= 90 output=  5 | train=0.004105 val=0.004166 test=0.005806 epochs=200 params=219523


input= 90 output= 30 | train=0.001864 val=0.001768 test=0.002427 epochs=200 params=219523


input= 90 output= 90 | train=0.001332 val=0.001036 test=0.001387 epochs=200 params=219523


,model_name,input_window,output_window,MAE_train,MAE_val,MAE_test,epochs,n_params
0,mlp_2x100_dropout,5,1,0.010278,0.009028,0.012710,200,24023
1,mlp_2x100_dropout,5,5,0.004771,0.004166,0.005774,200,24023
2,mlp_2x100_dropout,5,30,0.002081,0.001764,0.002417,200,24023
3,mlp_2x100_dropout,5,90,0.001277,0.001023,0.001414,200,24023
4,mlp_2x100_dropout,10,1,0.009759,0.009017,0.012971,200,35523
5,mlp_2x100_dropout,10,5,0.004466,0.004161,0.005852,200,35523
6,mlp_2x100_dropout,10,30,0.001976,0.001758,0.002488,200,35523
7,mlp_2x100_dropout,10,90,0.001245,0.001027,0.001412,200,35523
8,mlp_2x100_dropout,30,1,0.009357,0.009019,0.012797,200,81523
9,mlp_2x100_dropout,30,5,0.004379,0.004162,0.005754,200,81523


## Matrices de error

In [5]:
mae_train_matrix = results_df.pivot(index="input_window", columns="output_window", values="MAE_train")
mae_val_matrix = results_df.pivot(index="input_window", columns="output_window", values="MAE_val")
mae_test_matrix = results_df.pivot(index="input_window", columns="output_window", values="MAE_test")

print("MAE train")
display(mae_train_matrix)
print("MAE val")
display(mae_val_matrix)
print("MAE test")
display(mae_test_matrix)


MAE train


output_window,1,5,30,90
input_window,,,,
5,0.010278,0.004771,0.002081,0.001277
10,0.009759,0.004466,0.001976,0.001245
30,0.009357,0.004379,0.001887,0.001187
90,0.008984,0.004105,0.001864,0.001332


MAE val


output_window,1,5,30,90
input_window,,,,
5,0.009028,0.004166,0.001764,0.001023
10,0.009017,0.004161,0.001758,0.001027
30,0.009019,0.004162,0.001759,0.001036
90,0.009032,0.004166,0.001768,0.001036


MAE test


output_window,1,5,30,90
input_window,,,,
5,0.012710,0.005774,0.002417,0.001414
10,0.012971,0.005852,0.002488,0.001412
30,0.012797,0.005754,0.002463,0.001394
90,0.012829,0.005806,0.002427,0.001387


## Heatmaps

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
matrices = [
    (mae_train_matrix, "MAE train"),
    (mae_val_matrix, "MAE val"),
    (mae_test_matrix, "MAE test"),
]
vmin = min(matrix.values.min() for matrix, _ in matrices)
vmax = max(matrix.values.max() for matrix, _ in matrices)

for ax, (matrix, title) in zip(axes, matrices):
    im = ax.imshow(matrix.values, cmap="viridis", aspect="auto", vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(matrix.columns)))
    ax.set_xticklabels(matrix.columns)
    ax.set_yticks(range(len(matrix.index)))
    ax.set_yticklabels(matrix.index)
    ax.set_xlabel("Ventana salida")
    ax.set_ylabel("Ventana entrada")
    ax.set_title(f"{title} - {MODEL_NAME}")
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f"{matrix.values[i, j]:.4f}", ha="center", va="center", color="white", fontsize=9)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()


/var/folders/py/c5_xfbqn469g5_844mv32gt40000gn/T/ipykernel_64583/2336802165.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Curvas de aprendizaje

In [7]:
fig, axes = plt.subplots(4, 4, figsize=(18, 14), sharey=False)
axes = axes.ravel()

for ax, ((in_w, out_w), group) in zip(
    axes,
    history_df.groupby(["input_window", "output_window"], sort=True),
):
    ax.plot(group["epoch"], group["loss"], label="train")
    ax.plot(group["epoch"], group["val_loss"], label="val")
    ax.set_title(f"in={in_w}, out={out_w}")
    ax.set_xlabel("epoch")
    ax.set_ylabel("MAE")
    ax.grid(True, alpha=0.3)

axes[0].legend()
plt.suptitle(f"Curvas de aprendizaje - {MODEL_NAME}", y=1.02)
plt.tight_layout()
plt.show()


/var/folders/py/c5_xfbqn469g5_844mv32gt40000gn/T/ipykernel_64583/1319896804.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Comparacion contra regresion lineal

In [8]:
comparison = compare_to_benchmark(results_df, benchmark="lr_benchmark")
display(comparison)

plot_benchmark_comparison(results_df, benchmark="lr_benchmark", model_name=MODEL_NAME)
plt.show()


,input_window,output_window,MAE_test,MAE_test_benchmark,delta,pct_delta
0,5,1,0.012710,0.012384,0.000326,2.631836
1,5,5,0.005774,0.005625,0.000150,2.659874
2,5,30,0.002417,0.002340,0.000077,3.273956
3,5,90,0.001414,0.001271,0.000143,11.220080
4,10,1,0.012971,0.012554,0.000417,3.320112
5,10,5,0.005852,0.005698,0.000155,2.713440
6,10,30,0.002488,0.002358,0.000130,5.493194
7,10,90,0.001412,0.001282,0.000130,10.120314
8,30,1,0.012797,0.012924,-0.000127,-0.985009
9,30,5,0.005754,0.005877,-0.000123,-2.095626


/var/folders/py/c5_xfbqn469g5_844mv32gt40000gn/T/ipykernel_64583/4288889828.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
